In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd

In [3]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        #структура нейросети: input_size - 128 - 64 - 32 - 1
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
traffic = pd.read_csv('../../data/df_train_traffic_encoded_scaled.csv')

In [5]:
traffic = traffic.drop(['Выручка'], axis=1)

In [6]:
#делим на 60000, потому что остальные переменные в районе -1 и 1, веса становятся слишком большими и результаты становятся нестабильными
traffic['Трафик'] = traffic['Трафик'] / 60000
traffic

,Трафик,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",month_sin,month_cos,pca_1,pca_2,pca_3,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0.994367,-0.183291,-0.634931,-0.658506,0.29525,-8.660254e-01,5.000000e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
1,0.944567,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
2,0.858133,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
3,0.944883,-0.183291,-0.634931,-0.658506,0.29525,1.224647e-16,-1.000000e+00,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
4,0.968800,-0.183291,-0.634931,-0.658506,0.29525,-5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232664,0.861267,-0.187406,-0.712807,-0.009117,0.52820,-8.660254e-01,5.000000e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232665,0.858600,-0.187406,-0.712807,-0.009117,0.52820,-5.000000e-01,8.660254e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232666,0.826550,-0.187406,-0.712807,-0.009117,0.52820,-1.000000e+00,-1.836970e-16,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232667,0.868583,-0.187406,-0.712807,-0.009117,0.52820,-2.449294e-16,1.000000e+00,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1


In [7]:
mlp = MLP(input_size=(traffic.shape[1] - 1))
#функци потерь, сочетающая RMSE и MAE
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

In [8]:
y_traffic = traffic['Трафик']
x_traffic = traffic.drop(['Трафик'], axis=1)

In [9]:
from sklearn.model_selection import train_test_split

x_train_traffic, x_test_traffic, y_train_traffic, y_test_traffic = train_test_split(x_traffic, y_traffic, test_size=0.2, random_state=598)

In [10]:
#pytorch требует данных в формате тензоров
x_tensor = torch.tensor(x_train_traffic.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [11]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=100, shuffle=True)

In [12]:
from torch.nn.functional import l1_loss

In [13]:
x_test_tensor = torch.tensor(x_test_traffic.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [14]:
for epoch in range(300):
    mlp.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        
        predictions = mlp(batch_x)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()

    mlp.eval()    
    with torch.no_grad():
        train_preds = mlp(x_tensor)
        rmse_train = torch.sqrt(torch.mean((train_preds - y_tensor) ** 2)).item() * 60000
        mae_train = l1_loss(train_preds * 60000, y_tensor * 60000).item()
        
        test_preds = mlp(x_test_tensor)
        rmse_test = torch.sqrt(torch.mean((test_preds - y_test_tensor) ** 2)).item() * 60000
        mae_test = l1_loss(test_preds * 60000, y_test_tensor * 60000).item()
        
    print(f"Эпоха {epoch}"
          f" Train RMSE: {rmse_train:7.2f}, MAE: {mae_train:7.2f}"
          f" Test RMSE: {rmse_test:7.2f}, MAE: {mae_test:7.2f}")

Эпоха 0 Train RMSE: 9919.71, MAE: 7673.66 Test RMSE: 9970.62, MAE: 7676.99
Эпоха 1 Train RMSE: 9658.91, MAE: 7305.40 Test RMSE: 9742.75, MAE: 7348.16
Эпоха 2 Train RMSE: 9521.07, MAE: 7313.92 Test RMSE: 9607.54, MAE: 7349.53
Эпоха 3 Train RMSE: 9442.98, MAE: 7266.95 Test RMSE: 9545.05, MAE: 7321.47
Эпоха 4 Train RMSE: 9281.26, MAE: 7026.15 Test RMSE: 9413.19, MAE: 7115.94
Эпоха 5 Train RMSE: 9039.47, MAE: 6888.82 Test RMSE: 9211.30, MAE: 6999.41
Эпоха 6 Train RMSE: 9071.89, MAE: 6880.61 Test RMSE: 9246.34, MAE: 7005.33
Эпоха 7 Train RMSE: 8782.67, MAE: 6675.48 Test RMSE: 8999.44, MAE: 6820.51
Эпоха 8 Train RMSE: 8695.13, MAE: 6592.67 Test RMSE: 8942.06, MAE: 6761.92
Эпоха 9 Train RMSE: 8438.12, MAE: 6417.62 Test RMSE: 8706.53, MAE: 6598.62
Эпоха 10 Train RMSE: 8203.59, MAE: 6255.99 Test RMSE: 8493.06, MAE: 6457.83
Эпоха 11 Train RMSE: 8207.02, MAE: 6314.49 Test RMSE: 8527.68, MAE: 6518.61
Эпоха 12 Train RMSE: 7937.89, MAE: 6057.94 Test RMSE: 8262.61, MAE: 6285.74
Эпоха 13 Train RMSE: 7